# 01 — Data Collection

**Goal:** Collect all AI-related research papers from ArXiv (Jan 2024 - present).

We query 8 CS categories using half-month time windows. The ArXiv API silently fails
when paginating past ~10,000 results (HTTP 500), so each window must stay under that
limit. Half-month splits keep all windows safely in the 2,000-8,000 range.

Each window is checkpointed to disk. If the process crashes, re-running picks up
where it left off.

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import from src/
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.collect import (  # noqa: E402
    CATEGORIES,
    END_DATE,
    START_DATE,
    collect_papers,
    get_monthly_windows,
)

In [2]:
print("Categories:", CATEGORIES)
print(
    f"Time range: {START_DATE.strftime('%Y-%m-%d')} → {END_DATE.strftime('%Y-%m-%d')}"
)

windows = get_monthly_windows(START_DATE, END_DATE)
print(f"\nHalf-month windows: {len(windows)}")
for start, end in windows:
    print(f"  {start.strftime('%Y-%m-%d')} → {end.strftime('%Y-%m-%d')}")

Categories: ['cs.AI', 'cs.LG', 'cs.CL', 'cs.CV', 'cs.RO', 'cs.NE', 'cs.MA', 'cs.IR']
Time range: 2024-01-01 → 2026-02-27

Half-month windows: 52
  2024-01-01 → 2024-01-16
  2024-01-16 → 2024-02-01
  2024-02-01 → 2024-02-16
  2024-02-16 → 2024-03-01
  2024-03-01 → 2024-03-16
  2024-03-16 → 2024-04-01
  2024-04-01 → 2024-04-16
  2024-04-16 → 2024-05-01
  2024-05-01 → 2024-05-16
  2024-05-16 → 2024-06-01
  2024-06-01 → 2024-06-16
  2024-06-16 → 2024-07-01
  2024-07-01 → 2024-07-16
  2024-07-16 → 2024-08-01
  2024-08-01 → 2024-08-16
  2024-08-16 → 2024-09-01
  2024-09-01 → 2024-09-16
  2024-09-16 → 2024-10-01
  2024-10-01 → 2024-10-16
  2024-10-16 → 2024-11-01
  2024-11-01 → 2024-11-16
  2024-11-16 → 2024-12-01
  2024-12-01 → 2024-12-16
  2024-12-16 → 2025-01-01
  2025-01-01 → 2025-01-16
  2025-01-16 → 2025-02-01
  2025-02-01 → 2025-02-16
  2025-02-16 → 2025-03-01
  2025-03-01 → 2025-03-16
  2025-03-16 → 2025-04-01
  2025-04-01 → 2025-04-16
  2025-04-16 → 2025-05-01
  2025-05-01 → 2025-05-

## Fetch Papers from ArXiv

This takes ~20 minutes — each window requires multiple API requests with a 3-second delay. If interrupted, re-run and already-fetched windows load from checkpoints.

In [3]:
output_dir = project_root / "data" / "raw"
df = collect_papers(output_dir=output_dir)

2026-02-27 23:16:27,967 - INFO - Collecting 52 windows of AI papers from ArXiv
2026-02-27 23:16:27,968 - INFO -   2024-01-01: found checkpoint, skipping
2026-02-27 23:16:27,996 - INFO -   2024-01-16: found checkpoint, skipping
2026-02-27 23:16:28,036 - INFO -   2024-02-01: found checkpoint, skipping
2026-02-27 23:16:28,078 - INFO -   2024-02-16: found checkpoint, skipping
2026-02-27 23:16:28,120 - INFO -   2024-03-01: found checkpoint, skipping
2026-02-27 23:16:28,164 - INFO -   2024-03-16: found checkpoint, skipping
2026-02-27 23:16:28,211 - INFO -   2024-04-01: found checkpoint, skipping
2026-02-27 23:16:28,251 - INFO -   2024-04-16: found checkpoint, skipping
2026-02-27 23:16:28,291 - INFO -   2024-05-01: found checkpoint, skipping
2026-02-27 23:16:28,326 - INFO -   2024-05-16: found checkpoint, skipping
2026-02-27 23:16:28,379 - INFO -   2024-06-01: found checkpoint, skipping
2026-02-27 23:16:28,426 - INFO -   2024-06-16: found checkpoint, skipping
2026-02-27 23:16:28,470 - INFO - 

In [4]:
output_path = output_dir / "arxiv_papers.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path} ({len(df):,} papers)")

Saved to /home/dino/dev/learning/iu-bsc/unsupervised-learning/arxiv-ai-trends/data/raw/arxiv_papers.csv (226,879 papers)


In [5]:
print(f"Shape: {df.shape}")
print(f"Date range: {df['published'].min()} to {df['published'].max()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nPapers per category (primary):\n{df['primary_category'].value_counts()}")

Shape: (226879, 8)
Date range: 2024-01-01 to 2026-02-26

Missing values:
arxiv_id            0
title               0
abstract            0
authors             0
primary_category    0
categories          0
published           0
updated             0
dtype: int64

Papers per category (primary):
primary_category
cs.CV               55598
cs.LG               50734
cs.CL               34780
cs.RO               16200
cs.AI               15396
                    ...  
physics.atom-ph         2
physics.pop-ph          1
math.GN                 1
q-bio.SC                1
physics.atm-clus        1
Name: count, Length: 139, dtype: int64


In [6]:
df["month"] = df["published"].str[:7]
print(df["month"].value_counts().sort_index())

month
2024-01     5654
2024-02     7715
2024-03     8204
2024-04     7025
2024-05     8082
2024-06     8646
2024-07     7683
2024-08     6506
2024-09     7670
2024-10    10313
2024-11     7401
2024-12     8314
2025-01     6626
2025-02     9223
2025-03     9988
2025-04     8020
2025-05    11979
2025-06     9997
2025-07     8832
2025-08     9408
2025-09    10584
2025-10    11491
2025-11     9812
2025-12     8597
2026-01     9641
2026-02     9468
Name: count, dtype: int64


In [7]:
df.head()

,arxiv_id,title,abstract,authors,primary_category,categories,published,updated,month
0,2602.23363v1,MediX-R1: Open Ended Medical Reinforcement Lea...,"We introduce MediX-R1, an open-ended Reinforce...","Sahal Shaji Mullappilly, Mohammed Irfan Kurpat...",cs.CV,cs.CV,2026-02-26,2026-02-26,2026-02
1,2602.22740v1,AMLRIS: Alignment-aware Masked Learning for Re...,Referring Image Segmentation (RIS) aims to seg...,"Tongfei Chen, Shuo Yang, Yuguang Yang, Linlin ...",cs.CV,"cs.CV, cs.AI",2026-02-26,2026-02-26,2026-02
2,2602.22723v1,Human Label Variation in Implicit Discourse Re...,There is growing recognition that many NLP tas...,"Frances Yung, Daniil Ignatev, Merel Scholman, ...",cs.CL,cs.CL,2026-02-26,2026-02-26,2026-02
3,2602.22724v1,AgentSentry: Mitigating Indirect Prompt Inject...,Large language model (LLM) agents increasingly...,"Tian Zhang, Yiwei Xu, Juan Wang, Keyan Guo, Xi...",cs.CR,"cs.CR, cs.AI",2026-02-26,2026-02-26,2026-02
4,2602.22727v1,HulluEdit: Single-Pass Evidence-Consistent Sub...,Object hallucination in Large Vision-Language ...,"Yangguang Lin, Quan Fang, Yufei Li, Jiachen Su...",cs.CV,cs.CV,2026-02-26,2026-02-26,2026-02


In [8]:
df = df.drop(columns=["month"])
output_path = output_dir / "arxiv_papers.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} papers to {output_path}")

Saved 226,879 papers to /home/dino/dev/learning/iu-bsc/unsupervised-learning/arxiv-ai-trends/data/raw/arxiv_papers.csv


## Summary

Collected **226,879** unique papers from 8 ArXiv CS categories (Jan 2024 – Feb 2026).

**Key decisions:**
- Half-month windows to avoid ArXiv API pagination limit (~10k results)
- 52 windows total, largest was 8,076 papers (well under limit)
- 139 distinct primary categories appear because we query by *any* matching category, not just primary
- Zero duplicates across windows (clean date-range splits)
- Zero missing values

**Next:** Exploratory data analysis in `02_eda.ipynb`